# Wafer Fourier Y

칩 단위 `.csv` 또는 comma-separated `.txt` 파일에서 wafer별 annulus Fourier y-value를 계산합니다. 입력 파일의 encoding은 UTF-8, CP949/EUC-KR, UTF-16 계열을 자동으로 처리합니다.

필수 입력 칼럼은 기본 설정 기준 `root_lot_id`, `wafer_id`, `item_id`, `chip_x_pos`, `chip_y_pos`, `y_value`, `last_update_time`입니다. 사용자가 입력한 `item_id` 값, 예: `MSR0022`, 에 해당하는 row만 계산합니다. 같은 wafer에 여러 update snapshot이 있으면 `last_update_time`이 가장 최신인 chip row만 사용합니다.

## 계산식

annulus 영역 `r_in <= r <= r_out` 안에서 theta 방향으로 `y_value`를 평균내어 1D signal을 만듭니다.

$$
y(\theta_j) = \mathrm{mean}\{y_i \mid \theta_i \in \mathrm{bin}_j,\ r_i \in [r_{in}, r_{out}]\}
$$

평균 offset, 즉 DC 성분을 제거한 뒤 target harmonic `k`의 Fourier amplitude를 계산합니다.

$$
A_k = 2\left|\frac{1}{N}\sum_j (y(\theta_j)-\bar{y})e^{-ik\theta_j}\right|
$$

이 notebook의 기본 target은 `k = 16`입니다. 출력의 `fourier_y_value`는 선택한 target harmonic amplitude입니다.

$$
\mathrm{fourier\_y\_value} = A_{target}
$$

`signal_to_noise`는 target Fourier amplitude를 theta signal의 전체 AC variation으로 나눈 값입니다.

$$
\mathrm{signal\_to\_noise} = \frac{A_{target}}{\mathrm{std}(y(\theta)) + 10^{-12}}
$$

실무 score로는 아래 값을 함께 출력합니다.

$$
\mathrm{snr\_weighted\_fourier\_y} = \mathrm{fourier\_y\_value} \times \mathrm{signal\_to\_noise}
$$

In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy==1.26.4",
    "polars": "polars==1.14.0",
    "matplotlib": "matplotlib==3.9.2",
}

missing = [spec for module, spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required packages are already installed.")

## User Inputs

보통 아래 값만 바꾸면 됩니다. `INPUT_FILE`은 `.csv` 또는 comma-separated `.txt` 파일 경로이고, `ITEM_ID`는 `item_id` 칼럼에 들어있는 string 값입니다.

In [ ]:
from pathlib import Path

INPUT_FILE = Path("input.txt")
OUTPUT_CSV = Path("fourier_y_output.csv")
ITEM_ID = "MSR0022"
WAFER_TO_PLOT = ("ABCDE", 10)

INNER_RADIUS = 0.6
OUTER_RADIUS = 1.0
TARGET_HARMONIC = 16
ANGULAR_BINS = 384
MIN_RING_CHIPS = 20

# 실제 chip pitch를 알고 있으면 숫자로 지정하세요. 모르면 None으로 둡니다.
PITCH_X = None
PITCH_Y = None

# 최신 snapshot 필터가 필요 없으면 None으로 바꾸세요.
LAST_UPDATE_COL = "last_update_time"

## Run

아래 셀은 `fourier_y.py` 안의 로직을 호출합니다. 결과는 `OUTPUT_CSV`로 저장되고, `snr_weighted_fourier_y` 기준 내림차순으로 정렬됩니다.

In [ ]:
from fourier_y import FourierConfig, run_fourier_y

config = FourierConfig(
    input_path=INPUT_FILE,
    output_csv=OUTPUT_CSV,
    inner_radius=INNER_RADIUS,
    outer_radius=OUTER_RADIUS,
    target_harmonic=TARGET_HARMONIC,
    item_id_value=ITEM_ID,
    angular_bins=ANGULAR_BINS,
    min_ring_chips=MIN_RING_CHIPS,
    pitch_x=PITCH_X,
    pitch_y=PITCH_Y,
    update_col=LAST_UPDATE_COL,
)

run = run_fourier_y(config, write_csv=True)
result = run.result
analysis_df = run.polar_df

print(f"saved: {OUTPUT_CSV}")
print(f"rows={result.height:,}, columns={len(result.columns):,}")
print(f"analysis_df rows={analysis_df.height:,}, columns={len(analysis_df.columns):,}")
result.head(20)

## 주요 출력

- `fourier_y_value`: target harmonic amplitude입니다.
- `signal_to_noise`: `fourier_y_value / ring_std_y`입니다.
- `snr_weighted_fourier_y`: `fourier_y_value * signal_to_noise`입니다.
- `ring_mean_y`, `ring_std_y`: annulus theta signal의 평균과 표준편차입니다.
- `ring_chip_count`, `ring_coverage`: 계산에 사용된 annulus chip 수와 coverage입니다.
- `analysis_df`: `item_id` 필터와 latest snapshot 필터가 적용되고 `_theta`, `_radius_norm`이 추가된 chip-level Polars DataFrame입니다.

In [ ]:
summary_cols = [
    "root_lot_id",
    "wafer_id",
    "item_id",
    "fourier_y_value",
    "signal_to_noise",
    "snr_weighted_fourier_y",
    "ring_mean_y",
    "ring_std_y",
    "ring_chip_count",
    "ring_coverage",
]
result.select([col for col in summary_cols if col in result.columns]).head(30)

## Selected Wafer Charts

`WAFER_TO_PLOT = (root_lot_id, wafer_id)` 형태로 지정한 wafer 하나에 대해 annulus `mean_y_value` vs `theta [rad]` 차트와 계산 가능한 모든 harmonic amplitude 차트를 확인합니다. `ANGULAR_BINS = 384`이면 harmonic 1부터 192까지 계산합니다.

In [ ]:
from fourier_y import build_wafer_theta_signal, compute_harmonic_spectrum, max_calculable_harmonic

theta_signal_df = build_wafer_theta_signal(
    analysis_df,
    WAFER_TO_PLOT,
    group_cols=config.group_cols,
    y_col=config.value_col,
    inner_radius=INNER_RADIUS,
    outer_radius=OUTER_RADIUS,
    angular_bins=ANGULAR_BINS,
)

harmonic_spectrum_df = compute_harmonic_spectrum(
    theta_signal_df,
    angular_bins=ANGULAR_BINS,
    max_harmonic=max_calculable_harmonic(ANGULAR_BINS),
)

print(f"selected wafer: {WAFER_TO_PLOT}")
print(f"theta bins with chips: {theta_signal_df.height:,}")
theta_signal_df.head(20)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(
    theta_signal_df.get_column("theta_rad"),
    theta_signal_df.get_column("mean_y_value"),
    marker="o",
    markersize=2,
    linewidth=1,
)
ax.set_title(f"Mean y_value in annulus vs theta: {WAFER_TO_PLOT}")
ax.set_xlabel("theta [rad]")
ax.set_ylabel("mean_y_value in annulus")
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(
    harmonic_spectrum_df.get_column("harmonic"),
    harmonic_spectrum_df.get_column("amplitude"),
    width=0.9,
)
ax.axvline(TARGET_HARMONIC, color="red", linestyle="--", linewidth=1, label=f"target harmonic {TARGET_HARMONIC}")
ax.set_title(f"Harmonic amplitude spectrum: {WAFER_TO_PLOT}")
ax.set_xlabel("harmonic / frequency [cycles per revolution]")
ax.set_ylabel("amplitude")
ax.set_xlim(0, max_calculable_harmonic(ANGULAR_BINS) + 1)
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.show()

harmonic_spectrum_df.head(30)